# Notebook 9: Advanced Agents & LangGraph

**🎯 Goal:** Learn advanced agent architectures like Plan-and-Execute and use LangGraph to build powerful, stateful, and cyclic workflows for complex tasks.

## 🧩 Concept Overview

Simple ReAct agents are great, but what about very complex tasks like "write a comprehensive report on climate change"? A single `Thought -> Action` loop isn't enough. We need more sophisticated architectures.

- **Plan-and-Execute:** This architecture splits the task between two agents: a **Planner** and an **Executor**. The Planner breaks the main goal into a step-by-step plan. The Executor then carries out each step sequentially. It's great for tasks that can be clearly outlined in advance.

- **LangGraph:** This is a powerful library for building agents as **state machines**. It gives you fine-grained control to create any workflow you can imagine, including **cycles** (loops). This allows an agent to do things like `plan -> execute -> critique -> (if critique fails) -> re-plan`. It's the most flexible and powerful way to build agents.

## 🖼️ Visual Diagrams

### Plan-and-Execute Architecture
```
            +-----------+      +--------------------------------------+
User Query => |  Planner  | ===> | Plan: [Step 1, Step 2, Step 3, ...]  |
            +-----------+      +------------------+-----------------+
                                                  |
                                                  v
            +-----------+      +------------------+-----------------+
            |  Executor | <=== | Execute each step one by one       |
            +-----------+      +--------------------------------------+
```

### LangGraph Cyclic Architecture (Research & Write)
```
                 +-----------+       No, needs more research
                 | Researcher| <------------------------------------+
                 +-----+-----+                                      |
                       |                                            |
                       v                                            |
                 +-----+-----+                                      | (Loop)
                 |   Writer  |                                      |
                 +-----+-----+                                      |
                       |                                            |
                       v                                            |
                 +-----+-----+                                      |
 (Conditional)   | Reviewer  | ------------------------------------+
      Edge ----> |(is it good?)| 
                 +-----+-----+ 
                       |
                       | Yes, publish
                       v
                 +-----END-----+
```

## 1. Plan-and-Execute Agent

This pattern is useful for long-running tasks that require maintaining focus on a high-level objective. *Note: This API is still experimental and may change.*

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import Tool
from langchain_experimental.plan_and_execute import PlanAndExecute, load_agent_executor, load_chat_planner

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Define some tools for the executor
def search(query):
    return f"Mock search result for '{query}': The sky is blue due to Rayleigh scattering."
def web_scrape(url):
    return f"Mock scraped content from {url}: This is a long article about climate change..."

tools = [
    Tool(name="Search", func=search, description="useful for when you need to answer questions about current events"),
    Tool(name="WebScrape", func=web_scrape, description="useful for when you need to scrape a website for information")
]

# Load the planner and executor
planner = load_chat_planner(llm)
executor = load_agent_executor(llm, tools, verbose=True)

# Create the Plan-and-Execute agent
plan_and_execute_agent = PlanAndExecute(planner=planner, executor=executor, verbose=True)

query = "Write a short paragraph explaining why the sky is blue."
plan_and_execute_agent.invoke(query)

## 2. LangGraph for Cyclic Workflows

While Plan-and-Execute is powerful, it's not very flexible. LangGraph allows us to define custom, cyclic workflows where the agent can loop back to previous steps, reflect, and refine its work.

We'll rebuild the agent from Notebook 8, but this time we'll focus on the structure.

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Sequence
import operator
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolExecutor

load_dotenv()

@tool
def search_web(query: str) -> str:
    """Searches the web for the query."""
    print(f"-- Searching web for: {query} --")
    return f"LangGraph is a library for building stateful, multi-agent applications."

# The state defines the 'memory' of our graph
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# The nodes are the 'workers' in our graph
def call_model(state):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def call_tool(state):
    tool_calls = state["messages"][-1].tool_calls
    tool_messages = []
    for tc in tool_calls:
        output = tool_executor.invoke(tc)
        tool_messages.append(ToolMessage(content=str(output), tool_call_id=tc['id']))
    return {"messages": tool_messages}

# Conditional edges define the logic (the 'supervisor')
def should_continue(state):
    return "end" if not state["messages"][-1].tool_calls else "continue"

# Wire it all together
tools_list = [search_web]
tool_executor = ToolExecutor(tools_list)
llm_with_tools = llm.bind_tools(tools_list)

workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("action", call_tool)
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue, {"continue": "action", "end": END})
workflow.add_edge('action', 'agent') # This creates the loop!

app = workflow.compile()
print("LangGraph app compiled!")

## 🔧 Hands-On Exercise

**Goal:** Build a very simple two-agent LangGraph workflow: a Planner and a Summarizer.

1.  Define a state with two keys: `topic` (str) and `summary` (str).
2.  Create a `planner_node` that takes the initial state (with just a topic) and adds a `plan` string to the state (e.g., "The plan is to summarize the topic.").
3.  Create a `summarizer_node` that takes the state from the planner and adds a `summary` string (e.g., "This is a summary of [topic]").
4.  Create a graph, add the two nodes, and add an edge from the planner to the summarizer, then to END.
5.  Run the graph with a starting topic.

In [ ]:
class PlanState(TypedDict):
    topic: str
    plan: str
    summary: str

def planner_node(state: PlanState) -> PlanState:
    print("-- In Planner Node --")
    return {"plan": f"The plan is to write a summary about {state['topic']}."}

def summarizer_node(state: PlanState) -> PlanState:
    print("-- In Summarizer Node --")
    return {"summary": f"This is a mock summary about {state['topic']}. The plan was: {state['plan']}"}

exercise_graph = StateGraph(PlanState)
exercise_graph.add_node("planner", planner_node)
exercise_graph.add_node("summarizer", summarizer_node)
exercise_graph.set_entry_point("planner")
exercise_graph.add_edge("planner", "summarizer")
exercise_graph.add_edge("summarizer", END)

app_exercise = exercise_graph.compile()

result = app_exercise.invoke({"topic": "the rise of AI"})
print("\n--- Final State ---")
print(result)

## 🐞 Bug Bounty

A common bug is creating a graph that gets stuck in an infinite loop. The graph below has a `critique` node that is supposed to end the loop, but its conditional logic is broken—it *always* sends the workflow back to the `agent` node.

**Task:** Fix the `is_complete` function so that it returns `"end"` when the word `"final"` appears in the last message, correctly terminating the loop.

In [ ]:
# --- BROKEN CODE ---
def is_complete_buggy(state):
    # This logic is broken, it never ends!
    return "continue_to_agent"

# --- FIXED CODE ---
def is_complete_fixed(state):
    last_message_content = state["messages"][-1].content.lower()
    if "final answer" in last_message_content or "final response" in last_message_content:
        print("-- Critique: Work is complete. Ending loop. --")
        return "end"
    else:
        print("-- Critique: Work is not complete. Looping back. --")
        return "continue"

# We can imagine a graph like this:
# workflow.add_conditional_edges("agent", is_complete_fixed, {"continue": "agent", "end": END})
print("Bug fixed! The conditional edge logic now correctly identifies when to end the workflow.")

## 💡 Pro Tip

LangGraph is the future of building complex agents in LangChain. It is what powers many of the newest and most advanced agent executors under the hood. For any task that requires loops, reflection, or collaboration between multiple AI 'workers' (e.g., a researcher agent passing information to a writer agent), LangGraph is the right tool for the job. Mastering it is key to becoming an advanced agent engineer.

## 🏁 Summary

You've now explored the frontier of what's possible with AI agents.

1.  **Plan-and-Execute for Linear Tasks:** This architecture is effective for complex goals that can be broken down into a clear, linear sequence of steps.
2.  **LangGraph for Everything Else:** LangGraph's state machine model provides ultimate flexibility. It's the best way to build agents that need to loop, revise their work, or collaborate.
3.  **Graphs are Composed of Nodes and Edges:** You define the workers (nodes) and the workflow (edges), giving you complete control over the agent's behavior.